In [ ]:
# Databricks notebook source
# 60 TPD melter - SFC recommender: TRAIN + REGISTER IN MLFLOW (Unity Catalog) + MOVE THE SERVING ENDPOINT
#
# 1. reads the 15-min analytical record: Excel / CSV / Parquet in a Volume, or a Delta table
# 2. cleans it with the agreed rules (NCV validity and spikes, sensor zeros, draw doubling)
# 3. trains the gas recommender (daily gas-heat target -> NG, secondary air, AFR) and the
#    barrier-boost controller (MB3), plus the historical limits and the M&V yardsticks
# 4. checks the new model (signs, ranges, walk-forward calibration)
# 5. logs it to MLflow as a pyfunc model (code: sfc_pyfunc.py, numbers: model.json) and registers
#    a new version in Unity Catalog; if all checks pass it gets the alias "champion"
# 6. points the serving endpoint at the new champion version (an endpoint serves a version number,
#    it does not follow the alias by itself)
#
# Run it as a job (daily, after the previous day's SAP draw is available). Job parameters = the widgets.

# SFC Recommender — Training, MLflow Registration and Serving
**60 TPD flint melter · lines S09–S12 · melter natural gas, barrier boost, melter boost**

| | |
|---|---|
| **Reads** | the 15-min analytical record: `input_path` (`.xlsx` sheet `result`, `.csv` or `.parquet`, e.g. in a Volume) or `source_table` (Delta) |
| **Registers** | Unity Catalog model `uc_model_name`: one new version per run; alias **`champion`** = released model, **`previous`** = the one before (rollback) |
| **Serves** | endpoint `endpoint_name` is moved to the new champion version (skipped while the parameter is empty) |
| **Needs** | `sfc_pyfunc.py` in the same workspace folder as this notebook |

**What is trained** (same rules and numbers as `client/databricks/01_training_sfc_model`)
- **Gas recommender:** daily **gas-heat target** = regression on all usable history (draw, barrier boost, melter boost, furnace age), shifted to the level the most efficient quarter of the last 45 days reached. Every reversal cycle: `NG = target ÷ 96 ÷ latest NCV`; `secondary air = air-per-Mcal target × gas heat`; `AFR = air ÷ NG`.
- **Barrier-boost recommender:** hourly controller on the MB3 bottom temperature: `boost next hour = boost last hour + 0.1 × (MB3 target − MB3 now) ÷ gain`.
- **Historical limits** (P1–P99): every setpoint is held inside them.

In [ ]:
%pip install -q "mlflow>=2.16" openpyxl "databricks-sdk>=0.30"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import json
import os
import sys
import warnings
import datetime as dt
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mlflow
from mlflow import MlflowClient
from mlflow.models import ModelSignature, infer_signature
from mlflow.types import ColSpec, Schema

ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
print("mlflow", mlflow.__version__, "| pandas", pd.__version__, "| on Databricks:", ON_DATABRICKS)

## 1. Settings
Every setting a job may change is a **widget** (= job parameter). Defaults point at the Volume layout below; change them in the job, not in the code.

| Parameter | Meaning |
|---|---|
| `input_path` | the analytical record file. `.xlsx` (sheet `excel_sheet`), `.csv` or `.parquet` |
| `source_table` | a Delta table with the same columns. **When set, it is used instead of `input_path`** |
| `uc_model_name` | `catalog.schema.model` in Unity Catalog |
| `experiment_path` | MLflow experiment for the training runs |
| `endpoint_name` | serving endpoint to move to the new champion. Empty = don't touch any endpoint |

In [ ]:
def param(name: str, default: str) -> str:
    """Widget value in Databricks (job parameter), default elsewhere."""
    try:
        dbutils.widgets.text(name, default)          # noqa: F821
        return dbutils.widgets.get(name).strip()     # noqa: F821
    except NameError:
        return default


# ---- data -------------------------------------------------------------------------------------
INPUT_PATH = param("input_path", "/Volumes/prod_pgp_mfg/cantier/60_tpd_sfc/input/Analytical_Record_Creation_fixed_5.xlsx")
EXCEL_SHEET = param("excel_sheet", "result")
SOURCE_TABLE = param("source_table", "")         # e.g. prod_pgp_mfg.cantier.sfc_60tpd_analytical_record_15min

# ---- MLflow / Unity Catalog / serving ------------------------------------------------------
UC_MODEL_NAME = param("uc_model_name", "prod_pgp_mfg.cantier.sfc_60tpd_recommender")
EXPERIMENT_PATH = param("experiment_path", "/Shared/sfc_60tpd/recommender_training")
ENDPOINT_NAME = param("endpoint_name", "")        # e.g. sfc-60tpd-recommender ; empty = skip
CHAMPION, PREVIOUS = "champion", "previous"
MODEL_CODE = Path("sfc_pyfunc.py").resolve()      # next to this notebook

if not ON_DATABRICKS:                             # local test run: MLflow in a local folder
    LOCAL = Path(os.environ.get("SFC_LOCAL_MLFLOW", "mlflow_local")).resolve()
    LOCAL.mkdir(exist_ok=True)
    mlflow.set_tracking_uri(f"sqlite:///{LOCAL / 'mlflow.db'}")
    mlflow.set_registry_uri(f"sqlite:///{LOCAL / 'mlflow.db'}")
    INPUT_PATH = os.environ.get("SFC_INPUT_PATH", INPUT_PATH)
    UC_MODEL_NAME, EXPERIMENT_PATH = "sfc_60tpd_recommender", "sfc_60tpd_recommender_training"
else:
    mlflow.set_registry_uri("databricks-uc")

# ---- project constants (agreed with the plant; do not change without a review) ------------
MODEL_NAME = "sfc_recommender"
KCAL_PER_KWH = 860
AGE_ORIGIN = pd.Timestamp("2025-09-01")         # furnace-age clock for the ageing term
BASELINE_START, BASELINE_END = "2025-09-01", "2026-07-31"   # client baseline period (M&V, frozen)
CLIENT_BASELINE_SFC, CLIENT_TARGET_SFC = 1576.3, 1544.8

# ---- cleaning rules -------------------------------------------------------------------------
VALID_RANGE = {                                 # outside -> sensor fault / typing error -> missing
    "ncv": (8500, 10500), "opt_temp": (1550, 1600), "mb_kwh": (1, 200), "bb_kwh": (1, 250)}
NCV_SPIKE = 400                                 # kcal/SCM away from the ~2 h median -> glitch
DRAW_DOUBLED = 1.6                              # draw > 1.6 x surrounding-week median -> halved
MIN_COVERAGE = 0.90                             # a usable day needs >= 90% valid NCV, gas and boost

# ---- model settings (chosen in the model-selection study) -----------------------------------
GAS_FEATURES = ["draw_t", "bb_g", "mb_g", "age_m"]
SHIFT_WINDOW_DAYS, SHIFT_QUANTILE = 45, 0.25    # target = efficient quarter of the last 45 days
AIR_QUANTILE, AIR_FLOOR_QUANTILE = 0.25, 0.05
STEP_LAGS, KI = 12, 0.1                         # MB3 response horizon (h), controller speed
LIMIT_Q = (0.01, 0.99)                          # historical limits = P1-P99

print("source :", SOURCE_TABLE or INPUT_PATH)
print("model  :", UC_MODEL_NAME, "| experiment:", EXPERIMENT_PATH, "| endpoint:", ENDPOINT_NAME or "(not updated)")
assert MODEL_CODE.exists(), f"{MODEL_CODE} not found: put sfc_pyfunc.py next to this notebook"

## 2. Load the data
The analytical record with its original column names. The column map is the only place that knows them; if the source table uses other names, change the left-hand side only.

In [ ]:
COLUMNS = {
    "Timestamps": "ts",
    "BARRIER BOOSTER-52": "bb_kwh",                                   # kWh per 15 min
    "MELTER BOOSTER-51": "mb_kwh",                                    # kWh per 15 min
    "NCV Meter": "ncv",                                               # kcal/SCM
    "Cullet %": "cullet_pct",
    "Melter Optical Temperature": "opt_temp",                         # deg C (hourly, typed)
    "dcs_60_SecondaryAirFlowvalueFTSAFPVDB205DD92": "sec_air",        # 15-min mean of the 1-min DCS tag
    "dcs_60_MelterGasflowvalueFTMGFPVDB202DD92": "ng_scm",            # 15-min mean of the 1-min DCS tag
    "dcs_60_ThermocoupleMelterBottom3TCMB3PVDB249DD572": "mb3_temp",  # deg C
    "Seed Count": "seed_count",
    "Daily Draw": "draw_kg",                                          # SAP daily draw, kg
    "MB51_Quantity_KG (Draw)": "draw_kg",
}
NUMERIC = ["bb_kwh", "mb_kwh", "ncv", "cullet_pct", "opt_temp", "sec_air", "ng_scm", "mb3_temp", "seed_count", "draw_kg"]


def read_source() -> tuple[pd.DataFrame, str]:
    if SOURCE_TABLE:
        return spark.table(SOURCE_TABLE).toPandas(), SOURCE_TABLE          # noqa: F821
    p = INPUT_PATH.lower()
    if p.endswith((".xlsx", ".xlsm", ".xls")):
        return pd.read_excel(INPUT_PATH, sheet_name=EXCEL_SHEET), INPUT_PATH
    if p.endswith(".parquet"):
        return pd.read_parquet(INPUT_PATH), INPUT_PATH
    return pd.read_csv(INPUT_PATH), INPUT_PATH


raw, source = read_source()
raw = raw.rename(columns=COLUMNS)
missing = [c for c in ["ts"] + NUMERIC if c not in raw.columns]
if missing:
    raise ValueError(f"input is missing columns: {missing}")
raw["ts"] = pd.to_datetime(raw["ts"])
for c in NUMERIC:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
print(f"{source}: {len(raw):,} rows | {raw.ts.min():%d %b %Y %H:%M} to {raw.ts.max():%d %b %Y %H:%M}")
raw[["ts"] + NUMERIC].tail()

## 3. Clean the 15-minute data
| Rule | Why |
|---|---|
| one row per timestamp | duplicated rows are averaged |
| NCV kept only between 8,500 and 10,500 kcal/SCM | 0 = meter not working; real gas stays inside this range |
| NCV more than 400 away from the ~2 h median → missing | meter spikes |
| boosts and optical outside their plausible range → missing | dead sensors, typing errors |
| daily draw > 1.6 × the surrounding week → halved | two SAP entries added together on some month-start days |
| gas heat = Σ NG × NCV per 15 min | the client's SFC formula, never daily NG × average NCV |

In [ ]:
def clean_15min(raw: pd.DataFrame) -> pd.DataFrame:
    q = raw.groupby("ts")[NUMERIC].mean().sort_index()
    for col, (lo, hi) in VALID_RANGE.items():
        q.loc[q[col].notna() & ~q[col].between(lo, hi), col] = np.nan
    med = q["ncv"].rolling(9, center=True, min_periods=3).median()
    q.loc[(q["ncv"] - med).abs() > NCV_SPIKE, "ncv"] = np.nan
    day_draw = q["draw_kg"].groupby(q.index.normalize()).first()
    week_med = day_draw.rolling(7, center=True, min_periods=3).median()
    doubled = day_draw > DRAW_DOUBLED * week_med
    fixed = pd.Series(np.where(doubled, day_draw / 2, day_draw), index=day_draw.index)
    q["draw_fixed"] = doubled.reindex(q.index.normalize()).to_numpy()
    q["draw_kg"] = fixed.reindex(q.index.normalize()).to_numpy()
    q["afr"] = q["sec_air"] / q["ng_scm"]
    q["gas_kcal"] = q["ng_scm"] * q["ncv"]
    q.index.name = "ts"
    return q


q = clean_15min(raw)
fixed_days = sorted({t.date() for t in q.index[q["draw_fixed"].fillna(False).astype(bool)]})
print("NCV readings kept: %.1f%% | draw halved on: %s" % (100 * q.ncv.notna().mean(), ", ".join(map(str, fixed_days)) or "none"))

## 4. Daily and hourly tables

In [ ]:
def build_daily(q: pd.DataFrame) -> pd.DataFrame:
    f = q.copy()
    for col in ["ng_scm", "bb_kwh", "mb_kwh", "sec_air"]:
        f[col] = f[col].interpolate(limit=4, limit_area="inside")          # gaps <= 1 h only
    f["date"] = f.index.normalize()
    g = f.groupby("date")
    d = pd.DataFrame({
        "ng_scm": g["ng_scm"].sum(min_count=1), "sec_air": g["sec_air"].sum(min_count=1),
        "ng_cov": g["ng_scm"].apply(lambda s: s.notna().mean()), "ncv_cov": g["ncv"].apply(lambda s: s.notna().mean()),
        "bb_kwh": g["bb_kwh"].sum(min_count=1), "mb_kwh": g["mb_kwh"].sum(min_count=1),
        "boost_cov": g["bb_kwh"].apply(lambda s: s.notna().mean()),
        "draw_kg": g["draw_kg"].first(), "cullet_pct": g["cullet_pct"].first(), "seed_count": g["seed_count"].first(),
        "opt_temp": g["opt_temp"].mean(), "mb3_temp": g["mb3_temp"].mean()})
    both = f.dropna(subset=["ng_scm", "ncv"])
    d["ncv"] = (both["ng_scm"] * both["ncv"]).groupby(both["date"]).sum() / both.groupby("date")["ng_scm"].sum()
    d["gas_kcal"] = d["ng_scm"] * d["ncv"]
    d["energy_kcal"] = d["gas_kcal"] + (d["bb_kwh"] + d["mb_kwh"]) * KCAL_PER_KWH
    d["sfc"] = d["energy_kcal"] / d["draw_kg"]
    d["air_per_mcal"] = d["sec_air"] / (d["gas_kcal"] / 1000)
    d["draw_t"] = d["draw_kg"] / 1000
    d["usable"] = (d["ncv_cov"] >= MIN_COVERAGE) & (d["ng_cov"] >= MIN_COVERAGE) & (d["boost_cov"] >= MIN_COVERAGE) & d["draw_kg"].notna()
    d["in_baseline"] = (d.index >= BASELINE_START) & (d.index <= BASELINE_END)
    d.index.name = "date"
    return d


def build_hourly(q: pd.DataFrame) -> pd.DataFrame:
    r = q.resample("h")
    h = pd.DataFrame({"gas_kcal": r["gas_kcal"].sum(min_count=4), "bb_kwh": r["bb_kwh"].sum(min_count=4),
                      "mb_kwh": r["mb_kwh"].sum(min_count=4), "mb3_temp": r["mb3_temp"].mean()})
    h.index.name = "ts"
    return h


d, h = build_daily(q), build_hourly(q)
u = d[d.usable].copy()
u["gas_g"] = u["gas_kcal"] / 1e6
u["bb_g"] = u["bb_kwh"] * KCAL_PER_KWH / 1e6
u["mb_g"] = u["mb_kwh"] * KCAL_PER_KWH / 1e6
u["age_m"] = (u.index - AGE_ORIGIN).days / 30.44
u = u.dropna(subset=GAS_FEATURES + ["gas_g"])
print(f"usable days: {len(u)} of {len(d)} ({u.index.min():%d %b %Y} to {u.index.max():%d %b %Y}) | "
      f"baseline period: {int(u.in_baseline.sum())} | baseline SFC (our data): {u[u.in_baseline].sfc.mean():.1f} kcal/kg")

m = d.groupby(d.index.to_period("M")).agg(usable=("usable", "sum"), days=("usable", "size"))
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.bar(range(len(m)), m.usable, color="#2a78d6", width=0.7, label="usable days")
ax.bar(range(len(m)), m.days - m.usable, bottom=m.usable, color="#e1e0d9", width=0.7, label="left out")
ax.set_xticks(range(len(m))); ax.set_xticklabels([p.strftime("%b %y") for p in m.index], fontsize=8)
ax.set_title("Training data: usable days per month", loc="left", fontweight="bold"); ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

## 5. Train the gas recommender
`gas heat (Gcal/day) = a + b·draw + c·barrier boost + d·melter boost + e·furnace age`, fitted on all usable days, then shifted so that only the most efficient 25% of the last 45 days beat it.

In [ ]:
def ols(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    A = np.column_stack([np.ones(len(X)), X])
    return np.linalg.lstsq(A, y, rcond=None)[0]


def predict(beta: np.ndarray, X: np.ndarray) -> np.ndarray:
    return beta[0] + X @ beta[1:]


beta = ols(u[GAS_FEATURES].to_numpy(), u["gas_g"].to_numpy())
trained_until = u.index.max()
recent = u[u.index > trained_until - pd.Timedelta(days=SHIFT_WINDOW_DAYS)]
shift = float(np.quantile(recent["gas_g"] - predict(beta, recent[GAS_FEATURES].to_numpy()), SHIFT_QUANTILE))
gas_coefficients = dict(zip(["Intercept"] + GAS_FEATURES, map(float, beta)))

apm = u["air_per_mcal"].dropna()
air_per_mcal = float(max(apm[apm.index > trained_until - pd.Timedelta(days=SHIFT_WINDOW_DAYS)].quantile(AIR_QUANTILE),
                         apm.quantile(AIR_FLOOR_QUANTILE)))
print(f"efficient-quarter shift: {shift:+.3f} Gcal/day | air-per-Mcal target: {air_per_mcal:.4f} SCM/Mcal")
pd.DataFrame({"coefficient": gas_coefficients,
              "meaning": {"Intercept": "base level (Gcal/day)", "draw_t": "Gcal/day per +1 t/day draw",
                          "bb_g": "Gcal gas per Gcal barrier boost", "mb_g": "Gcal gas per Gcal melter boost",
                          "age_m": "Gcal/day per month of furnace age"}}).round(4)

## 6. Train the barrier-boost recommender
The MB3 response to a lasting +1 kWh/h of barrier boost, from the hourly history with a distributed-lag model (MB3 change vs boost and gas changes of the previous 0–12 h). The early dip caused by operator reactions is removed.

In [ ]:
x = h[["mb3_temp", "bb_kwh", "gas_kcal"]].dropna().asfreq("h")
lag = pd.DataFrame({"dmb3": x["mb3_temp"].diff()})
for k in range(STEP_LAGS + 1):
    lag[f"dbb{k}"] = x["bb_kwh"].diff().shift(k)
    lag[f"dgas{k}"] = (x["gas_kcal"] / 1e6).diff().shift(k)
lag = lag.dropna()
cols = [c for c in lag.columns if c != "dmb3"]
coef = pd.Series(ols(lag[cols].to_numpy(), lag["dmb3"].to_numpy())[1:], index=cols)
step = coef[[f"dbb{k}" for k in range(STEP_LAGS + 1)]].cumsum().clip(lower=0).cummax().reset_index(drop=True)
mb3_gain = float(step.iloc[-1])
mb3_target = float(h["mb3_temp"].median())
print(f"MB3 gain: {mb3_gain * 100:.2f} degC per +100 kWh/h ({len(lag):,} hours) | MB3 target (historical median): {mb3_target:.2f} degC")

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(step.index, step * 100, color="#2a78d6", marker="o", lw=2)
ax.set_xlabel("hours after a +100 kWh/h step"); ax.set_ylabel("MB3 change (degC)")
ax.set_title("MB3 response to barrier boost", loc="left", fontweight="bold"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Historical limits (P1–P99) and the M&V yardsticks

In [ ]:
lo, hi = LIMIT_Q
limits = {
    "ng_scm":       [q["ng_scm"].quantile(lo), q["ng_scm"].quantile(hi)],
    "afr":          [q["afr"].quantile(lo), q["afr"].quantile(hi)],
    "bb_kwh_h":     [h["bb_kwh"].quantile(lo), h["bb_kwh"].quantile(hi)],
    "mb3_temp":     [h["mb3_temp"].quantile(lo), h["mb3_temp"].quantile(hi)],
    "opt_temp":     [q["opt_temp"].quantile(lo), q["opt_temp"].quantile(hi)],
    "ncv":          [float(VALID_RANGE["ncv"][0]), float(VALID_RANGE["ncv"][1])],
    "draw_t":       [u["draw_t"].quantile(lo), u["draw_t"].quantile(hi)],
    "sec_air":      [q["sec_air"].quantile(lo), q["sec_air"].quantile(hi)],
    "gas_day_gcal": [u["gas_g"].quantile(lo), u["gas_g"].quantile(hi)],
}
limits = {k: [float(a), float(b)] for k, (a, b) in limits.items()}
LIMIT_LABELS = {"ng_scm": "NG setpoint (per 15 min)", "afr": "Air-fuel ratio", "bb_kwh_h": "Barrier boost (kWh/h)",
                "mb3_temp": "MB3 bottom temperature (degC)", "opt_temp": "Optical crown temperature (degC)",
                "ncv": "NCV accepted (kcal/SCM)", "draw_t": "Draw (t/day)", "sec_air": "Secondary air (per 15 min)",
                "gas_day_gcal": "Daily gas heat (Gcal/day, flag only)"}
lim_table = pd.DataFrame(limits, index=["lower limit (P1)", "upper limit (P99)"]).T
lim_table.insert(0, "item", [LIMIT_LABELS[k] for k in lim_table.index])

# M&V yardsticks: fitted on the fixed baseline period, so they never move on retraining
base = u[u.in_baseline]
mA = ols(base[["draw_t", "cullet_pct"]].to_numpy(), (base["energy_kcal"] / 1e6).to_numpy())
model_a = {"Intercept": float(mA[0]), "draw_t": float(mA[1]), "cullet_pct": float(mA[2])}
bands = pd.cut(base["draw_t"], [45, 50, 55, 60, 65], labels=["45-50", "50-55", "55-60", "60-65"], include_lowest=True)
band_baseline = {str(k): float(v) for k, v in base.groupby(bands, observed=True)["sfc"].mean().items()}
print("draw-adjusted yardstick: E = %.2f + %.3f x draw + %.3f x cullet (Gcal/day)" % tuple(model_a.values()))
lim_table.round(2)

## 8. Checks before the model is released
- **Signs and ranges:** the coefficients must make physical sense.
- **Walk-forward calibration:** for each of the last 30 usable days, the target is rebuilt from earlier days only. About 25% of days should already beat it.

The new version is **always registered** (for the record), but it only becomes **`champion`** — and only then reaches the endpoint — if every check passes.

In [ ]:
def walk_forward_target(day) -> float:
    hist = u[u.index < day]
    b = ols(hist[GAS_FEATURES].to_numpy(), hist["gas_g"].to_numpy())
    rec = hist[hist.index >= day - pd.Timedelta(days=SHIFT_WINDOW_DAYS)]
    s = np.quantile(rec["gas_g"] - predict(b, rec[GAS_FEATURES].to_numpy()), SHIFT_QUANTILE)
    return float(predict(b, u.loc[[day], GAS_FEATURES].to_numpy())[0] + s)


check_days = u.index[-30:]
wf = pd.DataFrame({"actual": u.loc[check_days, "gas_g"], "target": [walk_forward_target(t) for t in check_days]})
coverage = float((wf.actual <= wf.target).mean())

checks = {
    "usable training days >= 200":                        (len(u), len(u) >= 200),
    "draw coefficient > 0":                               (beta[1], beta[1] > 0),
    "barrier-boost coefficient between -1.2 and -0.3":    (beta[2], -1.2 < beta[2] < -0.3),
    "ageing coefficient >= 0":                            (beta[4], beta[4] >= 0),
    "MB3 gain 0.010-0.030 degC per kWh/h":                (mb3_gain, 0.010 < mb3_gain < 0.030),
    "air per Mcal 1.15-1.40":                             (air_per_mcal, 1.15 < air_per_mcal < 1.40),
    "last 30 days: share beating the target 10-45%":      (coverage, 0.10 <= coverage <= 0.45),
}
check_table = pd.DataFrame([{"check": k, "value": round(float(v), 4), "result": "PASS" if ok else "FAIL"} for k, (v, ok) in checks.items()])
passed = bool(all(ok for _, ok in checks.values()))
print("ALL CHECKS PASSED" if passed else "CHECK FAILED - the new version will NOT become champion")

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(wf.index, wf.target, color="#2a78d6", lw=2, label="gas target (from earlier days only)")
ax.scatter(wf.index, wf.actual, color=np.where(wf.actual <= wf.target, "#1baf7a", "#898781"), s=24, zorder=3, label="actual day")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_ylabel("gas heat (Gcal/day)"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=9)
ax.set_title(f"Last 30 days, walk-forward: {coverage:.0%} of days beat the target (25% intended)", loc="left", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
check_table

## 9. The model file
Only numbers (coefficients, targets, limits), same schema as `FinalRecommender.to_dict` / `models/recommender_latest.json`. It is packed into the MLflow model as the artifact `model_json`; the logic that uses it is `sfc_pyfunc.py`.

In [ ]:
created = dt.datetime.now()
tag = f"{trained_until:%Y-%m-%d}"
model = {
    "schema": 1,
    "model_name": MODEL_NAME,
    "model_version": f"{tag}_{created:%Y%m%d%H%M}",
    "created_at": created.isoformat(timespec="seconds"),
    "trained_until": tag,
    "n_training_days": int(len(u)),
    "data": {"source": source, "rows": int(len(raw)), "from": str(raw.ts.min()), "to": str(raw.ts.max())},
    "gas": {"formula": "gas_g ~ draw_t + bb_g + mb_g + age_m", "coefficients": gas_coefficients, "conformal_shift": shift,
            "age_origin": str(AGE_ORIGIN.date()), "days_per_month": 30.44, "intervals_per_day": 96, "kcal_per_kwh": KCAL_PER_KWH},
    "air": {"air_per_mcal": air_per_mcal},
    "boost": {"mb3_target": mb3_target, "mb3_gain_degC_per_kwh_h": mb3_gain, "ki": KI},
    "limits": limits,
    "mv": {"model_a": model_a, "band_baseline_sfc": band_baseline,
           "client_baseline_sfc": CLIENT_BASELINE_SFC, "client_target_sfc": CLIENT_TARGET_SFC},
}
report = {"model_version": model["model_version"], "trained_until": tag, "passed": passed,
          "checks": check_table.to_dict("records"), "walk_forward_coverage_last_30_days": coverage}

work = Path(tempfile.mkdtemp(prefix="sfc_model_"))
model_path = work / "model.json"
model_path.write_text(json.dumps(model, indent=2))
(work / "training_report.json").write_text(json.dumps(report, indent=2))
lim_table.to_csv(work / "historical_limits.csv")
print("model version", model["model_version"], "->", model_path)

## 10. Example request = the plant's latest values
The serving endpoint takes one row of live values. The example below is built from the last hours of the training data; it is stored with the model (so the endpoint UI shows a ready test request) and is used to check the logged model.

| Field | Source in the app |
|---|---|
| `timestamp` | now (IST) |
| `ncv` | latest valid NCV (UEMS MQTT) |
| `draw_t`, `cullet_pct`, `optical_temp` | typed by the operator |
| `bb_kwh_last_hour`, `mb_kwh_last_hour` | last hour's boosts (operator for now, DB later) |
| `mb3_now` | DCS, mean of the last 60 one-minute values |
| `ng_now`, `air_now` | DCS, mean of the last 15 one-minute values (optional, for "change") |

In [ ]:
def last_valid(s: pd.Series) -> float:
    s = s.dropna()
    return float(s.iloc[-1]) if len(s) else float("nan")


hb = q[["bb_kwh", "mb_kwh", "mb3_temp"]].resample("h").agg({"bb_kwh": "sum", "mb_kwh": "sum", "mb3_temp": "mean"})
hb = hb[q[["bb_kwh"]].resample("h").count()["bb_kwh"] == 4]
example = pd.DataFrame([{
    "timestamp": f"{q.index.max():%Y-%m-%d %H:%M}",
    "ncv": last_valid(q["ncv"]),
    "draw_t": round(last_valid(q["draw_kg"]) / 1000, 2),
    "cullet_pct": last_valid(q["cullet_pct"]),
    "bb_kwh_last_hour": round(last_valid(hb["bb_kwh"]), 1),
    "mb_kwh_last_hour": round(last_valid(hb["mb_kwh"]), 1),
    "mb3_now": round(last_valid(hb["mb3_temp"]), 2),
    "optical_temp": last_valid(q["opt_temp"]),
    "ng_now": round(last_valid(q["ng_scm"]), 2),
    "air_now": round(last_valid(q["sec_air"]), 1),
}])

# signature: required live values + optional ones (optical, current NG/air may be missing)
INPUT_SCHEMA = Schema(
    [ColSpec("string", "timestamp")]
    + [ColSpec("double", c) for c in ["ncv", "draw_t", "cullet_pct", "bb_kwh_last_hour", "mb_kwh_last_hour", "mb3_now"]]
    + [ColSpec("double", c, required=False) for c in ["optical_temp", "ng_now", "air_now"]])

sys.path.insert(0, str(MODEL_CODE.parent))
import sfc_pyfunc                                     # the model code, imported only to build the output schema
expected = pd.DataFrame([sfc_pyfunc.recommend_one(model, r) for r in example.to_dict("records")])
with warnings.catch_warnings():                       # integer-column hint: n_flags is never missing
    warnings.simplefilter("ignore", UserWarning)
    signature = ModelSignature(inputs=INPUT_SCHEMA, outputs=infer_signature(example, expected).outputs)
expected.T.rename(columns={0: "value"})

## 11. Log to MLflow and register in Unity Catalog
One MLflow run per training: parameters, metrics, the model file, the training report, the limits table and the **pyfunc model** (`sfc_pyfunc.py` + `model.json`). The model is registered as a new version of `uc_model_name`.

Aliases:
- **`champion`**: the released version. The app and the endpoint use this one.
- **`previous`**: the champion before it. Rollback = point `champion` back to it (Catalog Explorer → model → version → aliases, or `client.set_registered_model_alias`).

In [ ]:
mlflow.set_experiment(EXPERIMENT_PATH)
client = MlflowClient()
pip_requirements = [f"mlflow=={mlflow.__version__}", f"pandas=={pd.__version__}", f"numpy=={np.__version__}"]
where = {"name": "model"} if int(mlflow.__version__.split(".")[0]) >= 3 else {"artifact_path": "model"}   # MLflow 3 renamed it

with mlflow.start_run(run_name=f"{MODEL_NAME}_{tag}") as run:
    mlflow.log_params({"trained_until": tag, "n_training_days": len(u), "source": source[:250],
                       "shift_window_days": SHIFT_WINDOW_DAYS, "shift_quantile": SHIFT_QUANTILE, "ki": KI,
                       "gas_features": ",".join(GAS_FEATURES)})
    mlflow.log_metrics({"coverage_last_30d": coverage, "mb3_gain_degC_per_kwh_h": mb3_gain, "mb3_target": mb3_target,
                        "air_per_mcal": air_per_mcal, "conformal_shift": shift, "checks_passed": float(passed),
                        "baseline_sfc_our_data": float(u[u.in_baseline].sfc.mean()),
                        **{f"coef_{k}": v for k, v in gas_coefficients.items()}})
    mlflow.log_artifact(str(work / "training_report.json"))
    mlflow.log_artifact(str(work / "historical_limits.csv"))
    mlflow.log_dict(model, "model.json")
    info = mlflow.pyfunc.log_model(
        **where,
        python_model=str(MODEL_CODE),                 # models from code: no pickling
        artifacts={"model_json": str(model_path)},
        signature=signature,
        input_example=example,
        pip_requirements=pip_requirements,
        registered_model_name=UC_MODEL_NAME,
    )

version = str(getattr(info, "registered_model_version", "") or "")
if not version:                                       # older MLflow: look the version up by run id
    version = max((mv for mv in client.search_model_versions(f"name='{UC_MODEL_NAME}'") if mv.run_id == run.info.run_id),
                  key=lambda mv: int(mv.version)).version
for k, v in {"checks_passed": str(passed).lower(), "trained_until": tag, "model_version": model["model_version"],
             "coverage_last_30d": f"{coverage:.3f}"}.items():
    client.set_model_version_tag(UC_MODEL_NAME, version, k, v)
print(f"run {run.info.run_id} | registered {UC_MODEL_NAME} version {version}")

In [ ]:
def alias_version(alias: str):
    try:
        return client.get_model_version_by_alias(UC_MODEL_NAME, alias).version
    except Exception:
        return None


old_champion = alias_version(CHAMPION)
if passed:
    if old_champion and old_champion != version:
        client.set_registered_model_alias(UC_MODEL_NAME, PREVIOUS, old_champion)
    client.set_registered_model_alias(UC_MODEL_NAME, CHAMPION, version)
    print(f"version {version} is now '{CHAMPION}'" + (f" ('{PREVIOUS}' = version {old_champion})" if old_champion else ""))
else:
    print(f"checks failed: version {version} is registered but '{CHAMPION}' stays at version {old_champion}")
    print(check_table[check_table.result == "FAIL"].to_string(index=False))

## 12. Check the registered model
Loads the model back from the registry **exactly as the serving endpoint will** (`models:/<name>/<version>`), runs the example request and compares it with the numbers computed in this notebook. A serving-style payload is also validated.

In [ ]:
loaded = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_NAME}/{version}")
served = loaded.predict(example)
num_cols = [c for c in expected.columns if pd.api.types.is_numeric_dtype(expected[c]) and not pd.api.types.is_bool_dtype(expected[c])]
assert np.allclose(served[num_cols].to_numpy(float), expected[num_cols].to_numpy(float), equal_nan=True), "registered model differs"

payload = {"dataframe_records": json.loads(example.to_json(orient="records"))}
try:                                                  # MLflow >= 2.15: run the payload through the serving path
    from mlflow.models import validate_serving_input
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        validate_serving_input(f"models:/{UC_MODEL_NAME}/{version}", json.dumps(payload))
    print("serving payload validated")
except ImportError:
    pass
print("registered model reproduces the notebook. Example request for the endpoint:")
print(json.dumps(payload, indent=2))
served[["ng_setpoint", "air_setpoint", "afr_setpoint", "bb_setpoint_kwh_h", "gas_target_gcal_day", "sfc_expected", "flags"]]

## 13. Point the serving endpoint at the new champion
A Model Serving endpoint serves a **version number**; moving the `champion` alias does not redeploy it. So after a successful training this cell updates the endpoint to the champion version (zero-downtime roll-over; the old version keeps answering until the new one is ready).

- Runs only when `endpoint_name` is set **and** the checks passed.
- The endpoint must already exist (created once, see `Databricks/README.md`, step 3). The job's identity needs **Can Manage** on it.
- Workload size and scale-to-zero are copied from the current configuration.

In [ ]:
endpoint_status = "skipped (endpoint_name empty)" if not ENDPOINT_NAME else "skipped (checks failed)"
if ENDPOINT_NAME and passed and ON_DATABRICKS:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import ServedEntityInput

    w = WorkspaceClient()
    ep = w.serving_endpoints.get(ENDPOINT_NAME)
    current = (ep.config.served_entities or [None])[0] if ep.config else None
    if current is not None and current.entity_name == UC_MODEL_NAME and str(current.entity_version) == version:
        endpoint_status = f"already serving version {version}"
    else:
        w.serving_endpoints.update_config(
            name=ENDPOINT_NAME,
            served_entities=[ServedEntityInput(
                entity_name=UC_MODEL_NAME,
                entity_version=version,
                workload_size=(current.workload_size if current is not None and current.workload_size else "Small"),
                scale_to_zero_enabled=(bool(current.scale_to_zero_enabled) if current is not None else False),
            )],
        )
        endpoint_status = f"updating to version {version} (was {current.entity_version if current is not None else '-'})"
print("endpoint:", ENDPOINT_NAME or "-", "->", endpoint_status)

### Optional: query the endpoint
After the update has finished (Serving page → endpoint → *Ready*), this sends the example request to the endpoint, the same way the app does.

In [ ]:
QUERY_ENDPOINT = False
if QUERY_ENDPOINT and ENDPOINT_NAME and ON_DATABRICKS:
    from databricks.sdk import WorkspaceClient
    resp = WorkspaceClient().serving_endpoints.query(name=ENDPOINT_NAME, dataframe_records=payload["dataframe_records"])
    print(pd.DataFrame(resp.predictions).T)

## Summary

In [ ]:
pd.DataFrame({"value": {
    "trained until": tag, "usable training days": len(u), "checks": "PASS" if passed else "FAIL",
    "walk-forward coverage, last 30 days": f"{coverage:.0%}",
    "registered": f"{UC_MODEL_NAME} v{version}",
    "champion": alias_version(CHAMPION), "previous": alias_version(PREVIOUS),
    "endpoint": f"{ENDPOINT_NAME or '-'}: {endpoint_status}",
    "MLflow run": run.info.run_id}})